## Elastisearch - CRUD

Neste notbook, vamos fazer um teste simples utilizando duas bibliotecas: elastisearch e elastisearch_dsl

Links da documentação:

[elastisearch](https://elasticsearch-py.readthedocs.io/en/v8.17.1/)

[elastisearch_dsl](https://elasticsearch-dsl.readthedocs.io/en/latest/)

In [1]:
from elasticsearch import Elasticsearch, helpers
from elasticsearch_dsl import Document, Integer, Keyword, Text, Search,connections, Q
from typing import Optional

In [2]:
USERNAME = 'elastic'
PASSWORD =  'test10'
HOST = 'https://c099-2804-14d-8084-a528-31bb-b898-d2e9-3cc4.ngrok-free.app'

In [3]:
client = Elasticsearch(
    HOST,
    basic_auth=(USERNAME,PASSWORD)
)

### Verificando a conexão

In [4]:
client.info()

ObjectApiResponse({'name': 'elasticsearch', 'cluster_name': 'es-docker-cluster', 'cluster_uuid': 'M_MRJ2p6SLGZLngE4lpOig', 'version': {'number': '8.5.3', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '4ed5ee9afac63de92ec98f404ccbed7d3ba9584e', 'build_date': '2022-12-05T18:22:22.226119656Z', 'build_snapshot': False, 'lucene_version': '9.4.2', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

## CRUD

Para mostrar como utilizar as funcionalidades básicas das duas bibliotecas, será feito um CRUD (adicionar, ler, atualizar e deletar) os dados relacionados a músicas. Na célula abaixo, temos 3 músicas e suas propriedades.

In [8]:
levanta_anda = {
    "artista": "Emicida",
    "nome": "Levanta e Anda",
    "ano": 2014,
    "letra": """Era um cômodo incômodo
Sujo como o dragão-de-komodo, úmido
Eu, homem da casa aos seis anos
Mofo no canto todo, TV, engodo, pronto pro lodo
Tímido, porra, somos reis, mano
Olhos são eletrodos, sério, topo, trombo corvos
Num cemitério de sonhos graças a leis, planos
Troco de jogo, vendo, roubo
Pus a cabeça a prêmio, ingênuo
Colhi sorrisos e falei: Vamos
É um novo tempo, momento pro novo, ao sabor do vento
Eu me movo pelo solo onde reinamos
Pondo pontos finais na dor como Doril, Anador
Somos a luz do Senhor e pode crer
Tamo construindo, suponho, não, creio, meto a mão
Em meio à escuridão, pronto, acertamos
Nosso sorriso sereno hoje é o veneno
Pra quem trouxe tanto ódio pra onde deitamos

Quem costuma vir de onde eu sou
Às vezes não tem motivos pra seguir
Então levanta e anda
Vai, levanta e anda
Vai, levanta e anda
Mas eu sei que vai
Que o sonho te traz coisas que te faz prosseguir
Vai, levanta e anda
Vai, levanta e anda
Vai, levanta e anda
Vai, levanta e anda

Irmão
Você não percebeu que você é o único representante do seu sonho na face da Terra?
Se isso não fizer você correr, chapa
Eu não sei o que vai

Eu sei (sei)
Cansa
Quem morre ao fim do mês
Nossa grana ou nossa esperança?
Delírio é
Equilíbrio
Entre nosso martírio e nossa fé
Foi foda contar migalha nos escombro
Lona preta esticadas, enxada no ombro, e nada vim
Nada, enfim, recria sozinho
Com a alma cheia de mágoa e as panela vazia
Sonho imundo
Só água na geladeira e eu querendo salvar o mundo
No fundo, é tipo David Blaine
A mãe assume, o pai some, de costume
No máximo é um sobrenome
Sou o terror dos clone
Esses boy conhece Marx, nós conhece a fome
Então cerra os punho, sorria
E jamais volte pra sua quebrada de mão e mente vazia

Quem costuma vir de onde eu sou
Às vezes não tem motivos pra seguir
Então levanta e anda
Vai, levanta e anda
Vai, levanta e anda
Mas eu sei que vai
Que o sonho te traz coisas que te faz prosseguir
Então levanta e anda
Vai, levanta e anda
Vai, levanta e anda
Vai, levanta e anda

Somos maior
Nos basta só
Sonhar, seguir"""
}

hoje_cedo = {
    "artista": "Emicida",
    "nome": "Hoje é cedo",
    "ano": 2013,
    "letra":"""
    Hoje cedo
    Quando eu acordei e não te vi
    Eu pensei em tanta coisa
    Tive medo
    Ah, como eu chorei
    Eu sofri em segredo
    Tudo isso hoje cedo
    Holofotes fortes, purpurina
    E o sorriso dessas mina só me lembra cocaína
    Em cinco abrem-se cortinas
    Estáticas retinas brilham, garoa fina
    Que fita, meus poema me trouxe onde eles não habita
    A fama irrita, a grana dita, 'cê desacredita?
    Fantoches, pique Celso Pitta mentem
    Mortos tipo meu pai, nem eu me sinto presente
    Aí, é rima que cês quer, toma, duas, três
    Farta pra infartar cada um de vocês
    Num abismo sem volta, de festa, ladainha
    Minha alma afunda igual minha família em casa, sozinha
    Entre putas como um cafetão, coisas que afetam
    Sintonia, como eu sonhei em tá aqui um dia?
    Crise, trampo, ideologia, pause
    E é aqui onde nóis entende a Amy Winehouse
    Hoje cedo
    Quando eu acordei e não te vi
    Eu pensei em tanta coisa
    Tive medo
    Ah, como eu chorei
    Eu sofri em segredo
    Tudo isso hoje cedo
    Vagabundo, a trilha
    É um precipício, penso o melhor
    Quero salvar o mundo, pois desisti da minha família
    E numa luta mais difícil a frustração vai ser menor
    Digno de dó, só o pó, vazio, comum
    Que já é moda no século XXI
    Blacks com voz sagaz gravada
    Contra vilões que sangram a quebrada
    Só que raps por nóiz, por paz, mais nada
    Me pôs nas Gerais, numa cela trancada
    Eu lembrei do Racionais, reflexão
    Aí, "os próprio preto num 'tá nem aí com isso, não"
    É um clichê romântico, triste
    Vai perceber, vai ver, se matou e o paraíso não existe
    Eu ainda sou o Emicida da Rinha
    Lotei casas do Sul ao Norte,
    Mas esvaziei a minha
    E vou por aí, Taliban
    Vendo os boy beber dois mês de salário da minha irmã
    Hennessys, avelãs, camarins, fãs, globais
    Mano, onde eles tavam há dez anos atrás?
    Showbiz como a regra diz, lek
    A sociedade vende Jesus, por que não ia vender rap?
    O mundo vai se ocupar com seu cifrão
    Dizendo que a miséria é quem carecia de atenção
    Hoje cedo
    Quando eu acordei e não te vi
    Eu pensei em tanta coisa
    Tive medo
    Ah, como eu chorei
    Eu sofri em segredo
    Tudo isso hoje cedo

    """
}


dias_gloria = {
    "artista": "Charlie Brown Jr.",
    "nome": "Dias de luta, dias de gloria",
    "ano": 2013,
    "letra":"""
    Dias De Luta, Dias De Glória
    (Canta comigo meu povo)
    Na minha vida tudo acontece
    Mas quanto mais a gente rala, mais a gente cresce
    Hoje estou feliz porque eu sonhei com você
    E amanhã posso chorar por não poder te ver
    Mas o seu sorriso vale mais que um diamante
    Se você vier comigo, aí nós vamos adiante
    Com a cabeça erguida e mantendo a fé em Deus
    O seu dia mais feliz vai ser o mesmo que o meu
    A vida me ensinou a nunca desistir
    Nem ganhar, nem perder mas procurar evoluir
    Podem me tirar tudo que tenho
    Só não podem me tirar as coisas boas que eu já fiz pra quem eu amo
    E eu sou feliz e canto e o universo é uma canção
    E eu vou que vou
    História, nossas histórias
    Dias de luta, dias de glória
    História, nossas histórias
    Dias de luta, dias de glória
    História, nossas histórias
    Dias de luta, dias de glória
    História, nossas histórias
    Dias de luta, dias de glória
    Oh minha gata, morada dos meus sonhos
    Todo dia, se pudesse eu ia estar com você
    Eu já te via muito antes nos meus sonhos
    Eu procurei a vida inteira por alguém como você
    Por isso eu canto a minha vida com orgulho
    Com melodia, alegria e barulho
    Eu sou feliz e rodo pelo mundo
    Eu sou correria mas também sou vagabundo
    Mas hoje dou valor de verdade pra minha saúde,
    Pra minha liberdade
    Que bom te encontrar nessa cidade
    Esse brilho intenso me lembra você
    História, nossas histórias
    Dias de luta, dias de glória
    História, nossas histórias
    Dias de luta, dias de glória
    História, nossas histórias
    Dias de luta, dias de glória
    História, nossas histórias
    Dias de luta, dias de glória
    Hoje estou feliz, acordei com o pé direito
    E vou fazer de novo, vou fazer muito bem feito
    Sintonia, telepatia, comunicação pelo córtex bum bye bye
    """
}



## Inicializandoo processo de CRUD

Semelhante ao Mongodb ou Postgres, para adicionar dados no elastisearch, inicialmente é preciso informar onde será adicionado. O 'local' que será adicionado é chamado de index e ele possui as configurações de como cada chave do objeto deve ser indexado. (obs: nome do index precisa ser letra minúscula)

In [5]:
INDEX_LETRAS = 'letras_v2' 

In [15]:
if client.indices.exists(index=INDEX_LETRAS):
    client.indices.delete(index=INDEX_LETRAS)
client.indices.create(index=INDEX_LETRAS)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'letras_v2'})

### Adicionando a letra dentro do index

In [16]:
obj = client.index(
    index=INDEX_LETRAS,
    document=levanta_anda,
    id="levanta_e_anda"
)

In [17]:
obj

ObjectApiResponse({'_index': 'letras_v2', '_id': 'levanta_e_anda', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 0, '_primary_term': 1})

### Verificando como o index interpreta cada uma das chaves

In [18]:
client.indices.get(index=INDEX_LETRAS)

ObjectApiResponse({'letras_v2': {'aliases': {}, 'mappings': {'properties': {'ano': {'type': 'long'}, 'artista': {'type': 'text', 'fields': {'keyword': {'type': 'keyword', 'ignore_above': 256}}}, 'letra': {'type': 'text', 'fields': {'keyword': {'type': 'keyword', 'ignore_above': 256}}}, 'nome': {'type': 'text', 'fields': {'keyword': {'type': 'keyword', 'ignore_above': 256}}}}}, 'settings': {'index': {'routing': {'allocation': {'include': {'_tier_preference': 'data_content'}}}, 'number_of_shards': '1', 'provided_name': 'letras_v2', 'creation_date': '1740593946131', 'number_of_replicas': '1', 'uuid': 'Qau8Zry8TWWC5kY3IgQaKQ', 'version': {'created': '8050399'}}}}})

## Adicionando o objeto novamente

In [82]:
obj = client.index(
    index=INDEX_LETRAS,
    document=levanta_anda,
    id="levanta_e_anda"
)


In [83]:
obj

ObjectApiResponse({'_index': 'letras_v2', '_id': 'levanta_e_anda', '_version': 2, 'result': 'updated', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 1, '_primary_term': 1})

## Adicionando um conjunto

In [19]:
data = [
    {"_index":INDEX_LETRAS,"_id":"levanta_e_anda","_source":levanta_anda},
    {"_index":INDEX_LETRAS,"_id":"hoje_cedo","_source":hoje_cedo},
    {"_index":INDEX_LETRAS,"_id":"dias_gloria","_source":dias_gloria}
]

helpers.bulk(client, data)

(3, [])

In [20]:
client.get(index=INDEX_LETRAS,id="levanta_e_anda")

ObjectApiResponse({'_index': 'letras_v2', '_id': 'levanta_e_anda', '_version': 2, '_seq_no': 1, '_primary_term': 1, 'found': True, '_source': {'artista': 'Emicida', 'nome': 'Levanta e Anda', 'ano': 2014, 'letra': 'Era um cômodo incômodo\nSujo como o dragão-de-komodo, úmido\nEu, homem da casa aos seis anos\nMofo no canto todo, TV, engodo, pronto pro lodo\nTímido, porra, somos reis, mano\nOlhos são eletrodos, sério, topo, trombo corvos\nNum cemitério de sonhos graças a leis, planos\nTroco de jogo, vendo, roubo\nPus a cabeça a prêmio, ingênuo\nColhi sorrisos e falei: Vamos\nÉ um novo tempo, momento pro novo, ao sabor do vento\nEu me movo pelo solo onde reinamos\nPondo pontos finais na dor como Doril, Anador\nSomos a luz do Senhor e pode crer\nTamo construindo, suponho, não, creio, meto a mão\nEm meio à escuridão, pronto, acertamos\nNosso sorriso sereno hoje é o veneno\nPra quem trouxe tanto ódio pra onde deitamos\n\nQuem costuma vir de onde eu sou\nÀs vezes não tem motivos pra seguir\nEnt

In [21]:
client.get(index=INDEX_LETRAS,id="dias_gloria")

ObjectApiResponse({'_index': 'letras_v2', '_id': 'dias_gloria', '_version': 1, '_seq_no': 3, '_primary_term': 1, 'found': True, '_source': {'artista': 'Charlie Brown Jr.', 'nome': 'Dias de luta, dias de gloria', 'ano': 2013, 'letra': '\n    Dias De Luta, Dias De Glória\n    (Canta comigo meu povo)\n    Na minha vida tudo acontece\n    Mas quanto mais a gente rala, mais a gente cresce\n    Hoje estou feliz porque eu sonhei com você\n    E amanhã posso chorar por não poder te ver\n    Mas o seu sorriso vale mais que um diamante\n    Se você vier comigo, aí nós vamos adiante\n    Com a cabeça erguida e mantendo a fé em Deus\n    O seu dia mais feliz vai ser o mesmo que o meu\n    A vida me ensinou a nunca desistir\n    Nem ganhar, nem perder mas procurar evoluir\n    Podem me tirar tudo que tenho\n    Só não podem me tirar as coisas boas que eu já fiz pra quem eu amo\n    E eu sou feliz e canto e o universo é uma canção\n    E eu vou que vou\n    História, nossas histórias\n    Dias de lu

### Utilizando a biblioteca elastisearch_dsl para adicionar as letras


A biblioteca elastisearch_dsl conecta com o elastisearch através da biblioteca  elastisearch.

In [22]:
connections.add_connection("default",client)

In [23]:
INDEX_LETRAS2 = 'letras_dsl'

A biblioteca elastisearch_dsl permite criar uma classe que reflita o index do elastisearch, permitindo assim manipular as configurações
sem a necessidade de entender a linguagem do elastisearch previamente

In [24]:


class Letra(Document):
    artista : Optional[str] = Keyword()
    nome: Optional[str] = Keyword()
    ano:Optional[int] = Integer()
    letra:Optional[str] = Text()

    class Index:
        name = INDEX_LETRAS2

    def save(self, ** kwargs):
        return super().save(** kwargs)

### Criando o index

In [25]:
if Letra._index.exists():
    Letra._index.delete()
Letra.init()

In [26]:
levanta_anda_es = Letra(meta={"id": "levanta_e_anda"},**levanta_anda)

In [29]:
levanta_anda_es.letra

'Era um cômodo incômodo\nSujo como o dragão-de-komodo, úmido\nEu, homem da casa aos seis anos\nMofo no canto todo, TV, engodo, pronto pro lodo\nTímido, porra, somos reis, mano\nOlhos são eletrodos, sério, topo, trombo corvos\nNum cemitério de sonhos graças a leis, planos\nTroco de jogo, vendo, roubo\nPus a cabeça a prêmio, ingênuo\nColhi sorrisos e falei: Vamos\nÉ um novo tempo, momento pro novo, ao sabor do vento\nEu me movo pelo solo onde reinamos\nPondo pontos finais na dor como Doril, Anador\nSomos a luz do Senhor e pode crer\nTamo construindo, suponho, não, creio, meto a mão\nEm meio à escuridão, pronto, acertamos\nNosso sorriso sereno hoje é o veneno\nPra quem trouxe tanto ódio pra onde deitamos\n\nQuem costuma vir de onde eu sou\nÀs vezes não tem motivos pra seguir\nEntão levanta e anda\nVai, levanta e anda\nVai, levanta e anda\nMas eu sei que vai\nQue o sonho te traz coisas que te faz prosseguir\nVai, levanta e anda\nVai, levanta e anda\nVai, levanta e anda\nVai, levanta e anda

In [30]:
levanta_anda_es.save()

'created'

In [31]:
levanta_anda_es_return = Letra.get(id="levanta_e_anda")

In [32]:
levanta_anda_es_return.artista

'Emicida'

### Verificando a indexação

In [33]:
Letra._index.get_mapping()

ObjectApiResponse({'letras_dsl': {'mappings': {'properties': {'ano': {'type': 'integer'}, 'artista': {'type': 'keyword'}, 'letra': {'type': 'text'}, 'nome': {'type': 'keyword'}}}}})

## Executando leituras e pesquisas

In [34]:
levanta_anda_es = client.get(index=INDEX_LETRAS,id="levanta_e_anda")

In [35]:
levanta_anda_es

ObjectApiResponse({'_index': 'letras_v2', '_id': 'levanta_e_anda', '_version': 2, '_seq_no': 1, '_primary_term': 1, 'found': True, '_source': {'artista': 'Emicida', 'nome': 'Levanta e Anda', 'ano': 2014, 'letra': 'Era um cômodo incômodo\nSujo como o dragão-de-komodo, úmido\nEu, homem da casa aos seis anos\nMofo no canto todo, TV, engodo, pronto pro lodo\nTímido, porra, somos reis, mano\nOlhos são eletrodos, sério, topo, trombo corvos\nNum cemitério de sonhos graças a leis, planos\nTroco de jogo, vendo, roubo\nPus a cabeça a prêmio, ingênuo\nColhi sorrisos e falei: Vamos\nÉ um novo tempo, momento pro novo, ao sabor do vento\nEu me movo pelo solo onde reinamos\nPondo pontos finais na dor como Doril, Anador\nSomos a luz do Senhor e pode crer\nTamo construindo, suponho, não, creio, meto a mão\nEm meio à escuridão, pronto, acertamos\nNosso sorriso sereno hoje é o veneno\nPra quem trouxe tanto ódio pra onde deitamos\n\nQuem costuma vir de onde eu sou\nÀs vezes não tem motivos pra seguir\nEnt

## Pesquisas simples por termo


Link de como funciona a pesquisa no elastisearch

https://www.elastic.co/guide/en/elasticsearch/reference/current/analysis-overview.html

https://www.elastic.co/guide/en/elasticsearch/reference/current/analysis-overview.html

In [37]:
response = client.search(
    index=INDEX_LETRAS,
    query={
        "match":{
            "artista":"Emicida"
        }
    }
)

In [38]:
response['hits']

{'total': {'value': 2, 'relation': 'eq'},
 'max_score': 0.5619608,
 'hits': [{'_index': 'letras_v2',
   '_id': 'levanta_e_anda',
   '_score': 0.5619608,
   '_ignored': ['letra.keyword'],
   '_source': {'artista': 'Emicida',
    'nome': 'Levanta e Anda',
    'ano': 2014,
    'letra': 'Era um cômodo incômodo\nSujo como o dragão-de-komodo, úmido\nEu, homem da casa aos seis anos\nMofo no canto todo, TV, engodo, pronto pro lodo\nTímido, porra, somos reis, mano\nOlhos são eletrodos, sério, topo, trombo corvos\nNum cemitério de sonhos graças a leis, planos\nTroco de jogo, vendo, roubo\nPus a cabeça a prêmio, ingênuo\nColhi sorrisos e falei: Vamos\nÉ um novo tempo, momento pro novo, ao sabor do vento\nEu me movo pelo solo onde reinamos\nPondo pontos finais na dor como Doril, Anador\nSomos a luz do Senhor e pode crer\nTamo construindo, suponho, não, creio, meto a mão\nEm meio à escuridão, pronto, acertamos\nNosso sorriso sereno hoje é o veneno\nPra quem trouxe tanto ódio pra onde deitamos\n\nQu

In [39]:
response = client.search(
    index=INDEX_LETRAS,
    query={
        "match":{
            "artista":"emicida"
        }
    }
)

In [40]:
response['hits']

{'total': {'value': 2, 'relation': 'eq'},
 'max_score': 0.5619608,
 'hits': [{'_index': 'letras_v2',
   '_id': 'levanta_e_anda',
   '_score': 0.5619608,
   '_ignored': ['letra.keyword'],
   '_source': {'artista': 'Emicida',
    'nome': 'Levanta e Anda',
    'ano': 2014,
    'letra': 'Era um cômodo incômodo\nSujo como o dragão-de-komodo, úmido\nEu, homem da casa aos seis anos\nMofo no canto todo, TV, engodo, pronto pro lodo\nTímido, porra, somos reis, mano\nOlhos são eletrodos, sério, topo, trombo corvos\nNum cemitério de sonhos graças a leis, planos\nTroco de jogo, vendo, roubo\nPus a cabeça a prêmio, ingênuo\nColhi sorrisos e falei: Vamos\nÉ um novo tempo, momento pro novo, ao sabor do vento\nEu me movo pelo solo onde reinamos\nPondo pontos finais na dor como Doril, Anador\nSomos a luz do Senhor e pode crer\nTamo construindo, suponho, não, creio, meto a mão\nEm meio à escuridão, pronto, acertamos\nNosso sorriso sereno hoje é o veneno\nPra quem trouxe tanto ódio pra onde deitamos\n\nQu

In [41]:
analyze_query = {
    "analyzer": "standard",
    "text": "Emicida emicida"
}

client.indices.analyze(body=analyze_query)

ObjectApiResponse({'tokens': [{'token': 'emicida', 'start_offset': 0, 'end_offset': 7, 'type': '<ALPHANUM>', 'position': 0}, {'token': 'emicida', 'start_offset': 8, 'end_offset': 15, 'type': '<ALPHANUM>', 'position': 1}]})

### Pesquisar por keyword

In [42]:
client.search(
    index=INDEX_LETRAS,
    query={
        "match":{
            "artista.keyword":"emicida"
        }
    }
)

ObjectApiResponse({'took': 7, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 0, 'relation': 'eq'}, 'max_score': None, 'hits': []}})

In [43]:
client.search(
    index=INDEX_LETRAS,
    query={
        "match":{
            "letra":" cabeça a prêmio"
        }
    }
)

ObjectApiResponse({'took': 32, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 3, 'relation': 'eq'}, 'max_score': 1.6844437, 'hits': [{'_index': 'letras_v2', '_id': 'levanta_e_anda', '_score': 1.6844437, '_ignored': ['letra.keyword'], '_source': {'artista': 'Emicida', 'nome': 'Levanta e Anda', 'ano': 2014, 'letra': 'Era um cômodo incômodo\nSujo como o dragão-de-komodo, úmido\nEu, homem da casa aos seis anos\nMofo no canto todo, TV, engodo, pronto pro lodo\nTímido, porra, somos reis, mano\nOlhos são eletrodos, sério, topo, trombo corvos\nNum cemitério de sonhos graças a leis, planos\nTroco de jogo, vendo, roubo\nPus a cabeça a prêmio, ingênuo\nColhi sorrisos e falei: Vamos\nÉ um novo tempo, momento pro novo, ao sabor do vento\nEu me movo pelo solo onde reinamos\nPondo pontos finais na dor como Doril, Anador\nSomos a luz do Senhor e pode crer\nTamo construindo, suponho, não, creio, meto a mão\nEm meio à escuridão, pront

In [44]:
client.search(
    index=INDEX_LETRAS,
    query={
        "bool": {
            "must": [
                {"multi_match": {"query": "hoje"}}
            ]
        }
    }
)


ObjectApiResponse({'took': 28, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 3, 'relation': 'eq'}, 'max_score': 1.0925692, 'hits': [{'_index': 'letras_v2', '_id': 'hoje_cedo', '_score': 1.0925692, '_ignored': ['letra.keyword'], '_source': {'artista': 'Emicida', 'nome': 'Hoje é cedo', 'ano': 2013, 'letra': '\n    Hoje cedo\n    Quando eu acordei e não te vi\n    Eu pensei em tanta coisa\n    Tive medo\n    Ah, como eu chorei\n    Eu sofri em segredo\n    Tudo isso hoje cedo\n    Holofotes fortes, purpurina\n    E o sorriso dessas mina só me lembra cocaína\n    Em cinco abrem-se cortinas\n    Estáticas retinas brilham, garoa fina\n    Que fita, meus poema me trouxe onde eles não habita\n    A fama irrita, a grana dita, \'cê desacredita?\n    Fantoches, pique Celso Pitta mentem\n    Mortos tipo meu pai, nem eu me sinto presente\n    Aí, é rima que cês quer, toma, duas, três\n    Farta pra infartar cada um de vocês\n   

Verificando a diferença entre must, filter e should

In [46]:
response = client.search(
    index=INDEX_LETRAS,
    query={
       "bool":{
           "must":[
               {"match":{"artista":"Emicida"}},
               {"match":{"letra":"hoje"}},
           ]
       }
    }
)

In [49]:
response['hits']

{'total': {'value': 2, 'relation': 'eq'},
 'max_score': 0.80567735,
 'hits': [{'_index': 'letras_v2',
   '_id': 'hoje_cedo',
   '_score': 0.80567735,
   '_ignored': ['letra.keyword'],
   '_source': {'artista': 'Emicida',
    'nome': 'Hoje é cedo',
    'ano': 2013,
    'letra': '\n    Hoje cedo\n    Quando eu acordei e não te vi\n    Eu pensei em tanta coisa\n    Tive medo\n    Ah, como eu chorei\n    Eu sofri em segredo\n    Tudo isso hoje cedo\n    Holofotes fortes, purpurina\n    E o sorriso dessas mina só me lembra cocaína\n    Em cinco abrem-se cortinas\n    Estáticas retinas brilham, garoa fina\n    Que fita, meus poema me trouxe onde eles não habita\n    A fama irrita, a grana dita, \'cê desacredita?\n    Fantoches, pique Celso Pitta mentem\n    Mortos tipo meu pai, nem eu me sinto presente\n    Aí, é rima que cês quer, toma, duas, três\n    Farta pra infartar cada um de vocês\n    Num abismo sem volta, de festa, ladainha\n    Minha alma afunda igual minha família em casa, sozinh

In [53]:
response = client.search(
    index=INDEX_LETRAS,
    query={
        "bool": {
            "must": [
                {"match": {"artista": "Emicida"}},
            ],
            "filter":[
                  {"match": {"letra": "hoje"}}
            ]
        }
    }
)

In [54]:
response['hits']

{'total': {'value': 2, 'relation': 'eq'},
 'max_score': 0.5619608,
 'hits': [{'_index': 'letras_v2',
   '_id': 'levanta_e_anda',
   '_score': 0.5619608,
   '_ignored': ['letra.keyword'],
   '_source': {'artista': 'Emicida',
    'nome': 'Levanta e Anda',
    'ano': 2014,
    'letra': 'Era um cômodo incômodo\nSujo como o dragão-de-komodo, úmido\nEu, homem da casa aos seis anos\nMofo no canto todo, TV, engodo, pronto pro lodo\nTímido, porra, somos reis, mano\nOlhos são eletrodos, sério, topo, trombo corvos\nNum cemitério de sonhos graças a leis, planos\nTroco de jogo, vendo, roubo\nPus a cabeça a prêmio, ingênuo\nColhi sorrisos e falei: Vamos\nÉ um novo tempo, momento pro novo, ao sabor do vento\nEu me movo pelo solo onde reinamos\nPondo pontos finais na dor como Doril, Anador\nSomos a luz do Senhor e pode crer\nTamo construindo, suponho, não, creio, meto a mão\nEm meio à escuridão, pronto, acertamos\nNosso sorriso sereno hoje é o veneno\nPra quem trouxe tanto ódio pra onde deitamos\n\nQu

In [55]:
client.search(
    index=INDEX_LETRAS,
    _source=['artista','nome'],
    query={
        "bool": {
            "must": [
                {"match": {"letra": "hoje"}},
            ],
            "should":[
                  {"match":{"artista":"Charlie Brown "}}
            ]
        }
    }
)

ObjectApiResponse({'took': 7, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 3, 'relation': 'eq'}, 'max_score': 1.6943103, 'hits': [{'_index': 'letras_v2', '_id': 'dias_gloria', '_score': 1.6943103, '_ignored': ['letra.keyword'], '_source': {'artista': 'Charlie Brown Jr.', 'nome': 'Dias de luta, dias de gloria'}}, {'_index': 'letras_v2', '_id': 'hoje_cedo', '_score': 0.24371654, '_ignored': ['letra.keyword'], '_source': {'artista': 'Emicida', 'nome': 'Hoje é cedo'}}, {'_index': 'letras_v2', '_id': 'levanta_e_anda', '_score': 0.13160332, '_ignored': ['letra.keyword'], '_source': {'artista': 'Emicida', 'nome': 'Levanta e Anda'}}]}})

In [56]:
Letra.search().query('match', artista='Emicida').execute()

<Response: [Letra(index='letras_dsl', id='levanta_e_anda')]>

In [90]:
Letra.search().query('match', letra='Levanta e anda').execute()

<Response: [Letra(index='letras_dsl', id='levanta_e_anda')]>

In [ ]:
query = Q('bool',must=[Q('match',letra="hoje")],should=[Q('match',artista="Charlie Brown")])

In [59]:
Letra.search().query(query).execute()

<Response: [Letra(index='letras_dsl', id='levanta_e_anda')]>

## Atualizar os registros

In [60]:
client.update(index=INDEX_LETRAS, id="levanta_e_anda", doc={
    "ano": 2025
})

ObjectApiResponse({'_index': 'letras_v2', '_id': 'levanta_e_anda', '_version': 3, 'result': 'updated', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 4, '_primary_term': 1})

In [61]:
client.get(index=INDEX_LETRAS,id="levanta_e_anda")

ObjectApiResponse({'_index': 'letras_v2', '_id': 'levanta_e_anda', '_version': 3, '_seq_no': 4, '_primary_term': 1, 'found': True, '_source': {'artista': 'Emicida', 'nome': 'Levanta e Anda', 'ano': 2025, 'letra': 'Era um cômodo incômodo\nSujo como o dragão-de-komodo, úmido\nEu, homem da casa aos seis anos\nMofo no canto todo, TV, engodo, pronto pro lodo\nTímido, porra, somos reis, mano\nOlhos são eletrodos, sério, topo, trombo corvos\nNum cemitério de sonhos graças a leis, planos\nTroco de jogo, vendo, roubo\nPus a cabeça a prêmio, ingênuo\nColhi sorrisos e falei: Vamos\nÉ um novo tempo, momento pro novo, ao sabor do vento\nEu me movo pelo solo onde reinamos\nPondo pontos finais na dor como Doril, Anador\nSomos a luz do Senhor e pode crer\nTamo construindo, suponho, não, creio, meto a mão\nEm meio à escuridão, pronto, acertamos\nNosso sorriso sereno hoje é o veneno\nPra quem trouxe tanto ódio pra onde deitamos\n\nQuem costuma vir de onde eu sou\nÀs vezes não tem motivos pra seguir\nEnt

In [62]:
client.update(index=INDEX_LETRAS, id="levanta_e_anda", doc={
    "ano_shows": [2021,2022,2023]
})

ObjectApiResponse({'_index': 'letras_v2', '_id': 'levanta_e_anda', '_version': 4, 'result': 'updated', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 5, '_primary_term': 1})

In [63]:
client.get(index=INDEX_LETRAS,id="levanta_e_anda")

ObjectApiResponse({'_index': 'letras_v2', '_id': 'levanta_e_anda', '_version': 4, '_seq_no': 5, '_primary_term': 1, 'found': True, '_source': {'artista': 'Emicida', 'nome': 'Levanta e Anda', 'ano': 2025, 'letra': 'Era um cômodo incômodo\nSujo como o dragão-de-komodo, úmido\nEu, homem da casa aos seis anos\nMofo no canto todo, TV, engodo, pronto pro lodo\nTímido, porra, somos reis, mano\nOlhos são eletrodos, sério, topo, trombo corvos\nNum cemitério de sonhos graças a leis, planos\nTroco de jogo, vendo, roubo\nPus a cabeça a prêmio, ingênuo\nColhi sorrisos e falei: Vamos\nÉ um novo tempo, momento pro novo, ao sabor do vento\nEu me movo pelo solo onde reinamos\nPondo pontos finais na dor como Doril, Anador\nSomos a luz do Senhor e pode crer\nTamo construindo, suponho, não, creio, meto a mão\nEm meio à escuridão, pronto, acertamos\nNosso sorriso sereno hoje é o veneno\nPra quem trouxe tanto ódio pra onde deitamos\n\nQuem costuma vir de onde eu sou\nÀs vezes não tem motivos pra seguir\nEnt

In [64]:
levanta_anda_es_return = Letra.get(id="levanta_e_anda")

In [65]:
levanta_anda_es_return.ano = 2025

In [66]:
levanta_anda_es_return.save()

'updated'

In [67]:
Letra.get(id="levanta_e_anda").ano

2025

## Deletar registros

In [68]:
client.delete(index=INDEX_LETRAS, id="levanta_e_anda")

ObjectApiResponse({'_index': 'letras_v2', '_id': 'levanta_e_anda', '_version': 5, 'result': 'deleted', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 6, '_primary_term': 1})

In [69]:
client.get(index=INDEX_LETRAS,id="levanta_e_anda")

NotFoundError: NotFoundError(404, "{'_index': 'letras_v2', '_id': 'levanta_e_anda', 'found': False}")

In [70]:
levanta_anda_es_return = Letra.get(id="levanta_e_anda")

In [71]:
levanta_anda_es_return.delete()

In [72]:
Letra.get(id="levanta_e_anda")

NotFoundError: NotFoundError(404, "{'_index': 'letras_dsl', '_id': 'levanta_e_anda', 'found': False}")

## Topícos extras



Verificando as diferentes formas de pesquisar por texto

In [73]:
text_to_analyze = "Contra vilões que sangram a quebrada 2"


In [74]:
analyze_query = {
    "analyzer": "standard",
    "text": text_to_analyze
}


response = client.indices.analyze(body=analyze_query)

In [75]:
tokens = [token["token"] for token in response["tokens"]]
print("Tokens:", tokens)

Tokens: ['contra', 'vilões', 'que', 'sangram', 'a', 'quebrada', '2']


In [76]:
text_to_analyze = "Contra vilões que sangram a quebrada 2"

analyze_query = {
    "analyzer":"simple",
    "text": text_to_analyze
}


response = client.indices.analyze(body=analyze_query)

tokens = [token["token"] for token in response["tokens"]]
print("Tokens:", tokens)

Tokens: ['contra', 'vilões', 'que', 'sangram', 'a', 'quebrada']


In [77]:
text_to_analyze = "Contra vilões que sangram a quebrada 2"

analyze_query = {
    "analyzer":"portuguese",
    "text": text_to_analyze
}


response = client.indices.analyze(body=analyze_query)

tokens = [token["token"] for token in response["tokens"]]
print("Tokens:", tokens)

Tokens: ['contr', 'vila', 'sangram', 'quebrad', '2']


In [80]:
text_to_analyze = "Contras vilão que sangram a quebrado 2"

analyze_query = {
    "analyzer":"portuguese",
    "text": text_to_analyze
}


response = client.indices.analyze(body=analyze_query)

tokens = [token["token"] for token in response["tokens"]]
print("Tokens:", tokens)

Tokens: ['contr', 'vila', 'sangram', 'quebrad', '2']


In [83]:

analyze_query = {
    "tokenizer": "standard",  
    "filter": [
        "lowercase",  
        {
            "type": "shingle",  
            "min_shingle_size": 2,
            "max_shingle_size": 3,
            "output_unigrams": False 
        }
    ],
    "text": text_to_analyze
}

response = client.indices.analyze(body=analyze_query)

In [84]:
tokens = [token["token"] for token in response["tokens"]]
print("Tokens:", tokens)

Tokens: ['contras vilão', 'contras vilão que', 'vilão que', 'vilão que sangram', 'que sangram', 'que sangram a', 'sangram a', 'sangram a quebrado', 'a quebrado', 'a quebrado 2', 'quebrado 2']


Fazendo busca customizada

In [85]:
index_release = 'movie_release_v1'
movies = [
    {
        "name":"The Lion King",
        "anos":[1994,2011,2019]
    },
    {
        "name":"Star Wars: A New Hope",
        "anos":[1977,1997,2012]
    },
    {
        "name":"Jurassic Park",
        "anos":[1993,2013,2015]
    }
]

In [86]:
if client.indices.exists(index=index_release):
    client.indices.delete(index=index_release)
client.indices.create(index=index_release)

for movie in movies:
    client.index(index=index_release,document=movie)

In [149]:
Search(using=client, index=index_release).execute()

<Response: {}>

In [87]:
body = {
    "range": {
      "anos": {
        "gte": 2000
      }
    }
  }

client.search(
    index=index_release,
    query=body
)


ObjectApiResponse({'took': 8, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 3, 'relation': 'eq'}, 'max_score': 1.0, 'hits': [{'_index': 'movie_release_v1', '_id': 'jc-QQ5UBqNnXIfso0p7W', '_score': 1.0, '_source': {'name': 'The Lion King', 'anos': [1994, 2011, 2019]}}, {'_index': 'movie_release_v1', '_id': 'js-QQ5UBqNnXIfso055F', '_score': 1.0, '_source': {'name': 'Star Wars: A New Hope', 'anos': [1977, 1997, 2012]}}, {'_index': 'movie_release_v1', '_id': 'j8-QQ5UBqNnXIfso056W', '_score': 1.0, '_source': {'name': 'Jurassic Park', 'anos': [1993, 2013, 2015]}}]}})

In [88]:
body = {
    "bool":{
        "filter":{
            "script":{
                "script":{
                    "source":"""
                    if (doc['anos'].size() > 1){
                     return doc['anos'].get(1) < params.ano
                    }
                    else{
                        return false
                    }
                   
                    """,
                    "params":{
                        "ano":2000
                    }
                }
            }
        }
    }
}

client.search(
    index=index_release,
    query=body
)


ObjectApiResponse({'took': 199, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 1, 'relation': 'eq'}, 'max_score': 0.0, 'hits': [{'_index': 'movie_release_v1', '_id': 'js-QQ5UBqNnXIfso055F', '_score': 0.0, '_source': {'name': 'Star Wars: A New Hope', 'anos': [1977, 1997, 2012]}}]}})

In [89]:
body = {
    "bool":{
        "filter":{
            "script":{
                "script":{
                    "source":"""
                    if (doc['anos'].size() > 1){
                     return (doc['anos'].get(0) > params.ano_1 && doc['anos'].get(1) < params.ano_2)
                    }
                    else{
                        return false
                    }
                   
                    """,
                    "params":{
                        "ano_1":1900,
                        "ano_2":2000
                    }
                }
            }
        }
    }
}

client.search(
    index=index_release,
    query=body
)


ObjectApiResponse({'took': 45, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 1, 'relation': 'eq'}, 'max_score': 0.0, 'hits': [{'_index': 'movie_release_v1', '_id': 'js-QQ5UBqNnXIfso055F', '_score': 0.0, '_source': {'name': 'Star Wars: A New Hope', 'anos': [1977, 1997, 2012]}}]}})

Pesquisando por vetores criados por LLMs




[dense-vector](https://www.elastic.co/guide/en/elasticsearch/reference/current/dense-vector.html)

[vector-similarity](https://www.elastic.co/search-labs/blog/vector-similarity-techniques-and-scoring)

In [94]:
!pip install sentence-transformers

^C


  Obtaining dependency information for sentence-transformers from https://files.pythonhosted.org/packages/05/89/7eb147a37b7f31d3c815543df539d8b8d0425e93296c875cc87719d65232/sentence_transformers-3.4.1-py3-none-any.whl.metadata
  Using cached sentence_transformers-3.4.1-py3-none-any.whl.metadata (10 kB)
  Obtaining dependency information for transformers<5.0.0,>=4.41.0 from https://files.pythonhosted.org/packages/20/37/1f29af63e9c30156a3ed6ebc2754077016577c094f31de7b2631e5d379eb/transformers-4.49.0-py3-none-any.whl.metadata
  Using cached transformers-4.49.0-py3-none-any.whl.metadata (44 kB)
  Obtaining dependency information for torch>=1.11.0 from https://files.pythonhosted.org/packages/18/cf/ae99bd066571656185be0d88ee70abc58467b76f2f7c8bfeb48735a71fe6/torch-2.6.0-cp312-cp312-win_amd64.whl.metadata
  Using cached torch-2.6.0-cp312-cp312-win_amd64.whl.metadata (28 kB)
  Obtaining dependency information for scikit-learn from https://files.pythonhosted.org/packages/62/27/585859e72e117fe86


[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [95]:
from sentence_transformers import SentenceTransformer
model_name = 'sentence-transformers/all-MiniLM-L6-v2' # https://huggingface.co/models?pipeline_tag=sentence-similarity&sort=trending
model = SentenceTransformer(model_name)

c:\Users\spguilhermem\Documents\Pessoal\4_database_in_one_lecture\4databaseenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [96]:
sentences = levanta_anda['letra'].split("\n")

vector_denses = [{"sentence":sentence,"vector_sentence":model.encode(sentence)} for sentence in sentences]


In [97]:
len(vector_denses[0]['vector_sentence'])

384

#### Mapping the index to allow similarity

In [98]:
INDEX_LETRA_DENSE = 'letras_dense'

In [99]:
mappings = {
    "properties":{
        "vector_sentence":{
            "type":"dense_vector",
            "dims":384,
            "similarity":"cosine",
            "index":True
        },
        "sentence":{
            "type":"text"
        }
    }
}

In [100]:
if client.indices.exists(index=INDEX_LETRA_DENSE):
    client.indices.delete(index=INDEX_LETRA_DENSE)
client.indices.create(index=INDEX_LETRA_DENSE, body={"mappings":mappings})

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'letras_dense'})

In [101]:
for sentence_dense in vector_denses:
    client.index(index=INDEX_LETRA_DENSE, body=sentence_dense)

In [102]:
vector_pesquisa = model.encode("A mãe assumi, o pai desaparece, usual")

In [103]:
body = {
    "_source":['sentence'],
  "query": {
    "script_score": {
      "query": {
        "match_all": {}
      },
      "script": {
        "source": "cosineSimilarity(params.query_vector, 'vector_sentence') + 1.0",
        "params": {
          "query_vector": vector_pesquisa
        }
      }
    }
  },
  "size": 10 
}

client.search(
    index=INDEX_LETRA_DENSE,
    body=body
)

ObjectApiResponse({'took': 81, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 65, 'relation': 'eq'}, 'max_score': 1.69424, 'hits': [{'_index': 'letras_dense', '_id': 'ws-cQ5UBqNnXIfsofZ6K', '_score': 1.69424, '_source': {'sentence': 'A mãe assume, o pai some, de costume'}}, {'_index': 'letras_dense', '_id': 'vs-cQ5UBqNnXIfsofJ4m', '_score': 1.5780063, '_source': {'sentence': 'Com a alma cheia de mágoa e as panela vazia'}}, {'_index': 'letras_dense', '_id': 'uM-cQ5UBqNnXIfsoep4B', '_score': 1.5380613, '_source': {'sentence': 'Delírio é'}}, {'_index': 'letras_dense', '_id': 'ns-cQ5UBqNnXIfsocZ5b', '_score': 1.5334877, '_source': {'sentence': 'Tamo construindo, suponho, não, creio, meto a mão'}}, {'_index': 'letras_dense', '_id': 'mc-cQ5UBqNnXIfsob56l', '_score': 1.5240264, '_source': {'sentence': 'Colhi sorrisos e falei: Vamos'}}, {'_index': 'letras_dense', '_id': 'ks-cQ5UBqNnXIfsoap7p', '_score': 1.5156573, '_source':

In [199]:
body = {
     "_source":['sentence'],
    "knn":{
       "field":"vector_sentence",
       "query_vector":vector_pesquisa,
       "k":3,
       "num_candidates":100

    }
    
}

client.search(
    index=INDEX_LETRA_DENSE,
    body=body
)


ObjectApiResponse({'took': 22, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 3, 'relation': 'eq'}, 'max_score': 0.84712005, 'hits': [{'_index': 'letras_dense', '_id': 'OZIkNJUBt1DDIhktW3UX', '_score': 0.84712005, '_source': {'sentence': 'A mãe assume, o pai some, de costume'}}, {'_index': 'letras_dense', '_id': 'NZIkNJUBt1DDIhktWnV8', '_score': 0.78900325, '_source': {'sentence': 'Com a alma cheia de mágoa e as panela vazia'}}, {'_index': 'letras_dense', '_id': 'L5IkNJUBt1DDIhktWXWn', '_score': 0.7690307, '_source': {'sentence': 'Delírio é'}}]}})